# mdx2 profiling summary

In [ ]:
import os, platform, psutil

mem_bytes = psutil.virtual_memory().total
ncpu = os.cpu_count()
chip = platform.processor()  # or platform.machine()

print("Physical memory (GiB):", mem_bytes / 1024**3)
print("Number of CPU cores:", ncpu)
print("CPU:", chip)

## Insulin (Pilatus 6M, 100 frames)

### End-to-end tests

```bash
for n in 1 2 4 8; do ./run_profile.sh insulin_e2e.sh --nproc "$n"; done
```

In [ ]:
summary_files = "../runs/insulin_e2e-*/summary.log"

matches = (('Command line', "Command:"),
           ('Memory', "peak_tree_rss|peak_seen_at"),
           ('Wall time', "Overall wall time"),
           ('CPU percent est', "cpu_pct_est:"),
           ('mdx2.import_data log', "mdx2.import_data"),
           ('mdx2.find_peaks log', "mdx2.find_peaks"),
           ('mdx2.mask_peaks log', "mdx2.mask_peaks"),
           ('mdx2.integrate log', "mdx2.integrate"),
           ('mdx2.scale log', "mdx2.scale"),
           ('mdx2.merge log', "mdx2.merge"),
           ('mdx2.reintegrate log', "mdx2.reintegrate"))

for title, pattern in matches:
    # print the title (80 char width, centered)
    print(title.center(80, '-'))
    !grep -E "$pattern" $summary_files

### mdx2.integrate

```bash
for n in 1 2 4 8; do ./run_profile.sh insulin_integrate.sh --nproc "$n"; done
```

In [ ]:
summary_files = "../runs/insulin_integrate-*/summary.log"

matches = (('Command line', "Command:"),
           ('Memory', "peak_tree_rss|peak_seen_at"),
           ('Wall time', "Overall wall time"),
           ('CPU percent est', "cpu_pct_est:"),
           ('mdx2 Logs', "mdx2.integrate"))

for title, pattern in matches:
    # print the title (80 char width, centered)
    print(title.center(80, '-'))
    !grep -E "$pattern" $summary_files

### mdx2.import_data

```bash
for n in 1 2 4 8; do ./run_profile.sh insulin_import_data.sh --nproc "$n"; done
```

In [ ]:
summary_files = "../runs/insulin_import_data-*/summary.log"

matches = (('Command line', "Command:"),
           ('Memory', "peak_tree_rss|peak_seen_at"),
           ('Wall time', "Overall wall time"),
           ('CPU percent est', "cpu_pct_est:"),
           ('mdx2 Logs', "mdx2.import_data"))

for title, pattern in matches:
    # print the title (80 char width, centered)
    print(title.center(80, '-'))
    !grep -E "$pattern" $summary_files

### mdx2.find_peaks

```bash
for n in 1 2 4 8; do ./run_profile.sh insulin_find_peaks.sh --nproc "$n"; done
```

In [ ]:
summary_files = "../runs/insulin_find_peaks-*/summary.log"

matches = (('Command line', "Command:"),
           ('Memory', "peak_tree_rss|peak_seen_at"),
           ('Wall time', "Overall wall time"),
           ('CPU percent est', "cpu_pct_est:"),
           ('mdx2 Logs', "mdx2.find_peaks"))

for title, pattern in matches:
    # print the title (80 char width, centered)
    print(title.center(80, '-'))
    !grep -E "$pattern" $summary_files

### mdx2.mask_peaks

```bash
for n in 1 2 4 8; do ./run_profile.sh insulin_mask_peaks.sh --nproc "$n"; done
```

In [ ]:
summary_files = "../runs/insulin_mask_peaks-*/summary.log"

matches = (('Command line', "Command:"),
           ('Memory', "peak_tree_rss|peak_seen_at"),
           ('Wall time', "Overall wall time"),
           ('CPU percent est', "cpu_pct_est:"),
           ('mdx2 Logs', "mdx2.mask_peaks"))

for title, pattern in matches:
    # print the title (80 char width, centered)
    print(title.center(80, '-'))
    !grep -E "$pattern" $summary_files

### Peak memory comparison

In [ ]:
import re
from pathlib import Path
import pandas as pd

def load_summary_data(*tests, base="../runs"):
    rows = []
    for test in tests:
        for log in Path(base).glob(f"{test}-*/summary.log"):
            txt = log.read_text(errors="ignore")

            nproc = re.search(r"--nproc\s+(\d+)", txt)
            mem = re.search(r"peak_tree_rss_mib:\s*([0-9.]+)", txt)

            # CPU profile summary lines from run_profile.sh
            real_s = re.search(r"(?m)^real_s:\s*([0-9]+(?:\.[0-9]+)?)\s*$", txt)
            cpu_pct = re.search(r"(?m)^cpu_pct_est:\s*([0-9]+(?:\.[0-9]+)?)\s*$", txt)

            if nproc and mem:
                rows.append({
                    "test": test,
                    "nproc": int(nproc.group(1)),
                    "peak_tree_rss_mib": float(mem.group(1)),
                    "real_s": float(real_s.group(1)) if real_s else None,
                    "cpu_pct_est": float(cpu_pct.group(1)) if cpu_pct else None,
                    "log": str(log),
                })

    if not rows:
        return pd.DataFrame(columns=[
            "test", "nproc", "peak_tree_rss_mib", "real_s", "cpu_pct_est", "log"
        ])

    return pd.DataFrame(rows).sort_values(["test", "nproc"])

df = load_summary_data("insulin_import_data",  "insulin_find_peaks", "insulin_mask_peaks", "insulin_integrate", "dev_insulin_integrate")
df.head()

In [ ]:
ax = df.pivot_table(index="nproc", columns="test", values="peak_tree_rss_mib", aggfunc="mean").plot(marker="o")
ax.set_xlabel("nproc")
ax.set_ylabel("peak_tree_rss (MiB)")
ax.set_title("Peak memory for insulin dataset (Pilatus 6M, 100 frames)");

In [ ]:
ax = df.pivot_table(index="nproc", columns="test", values="cpu_pct_est", aggfunc="mean").plot(marker="o")
ax.set_xlabel("nproc")
ax.set_ylabel("CPU %")
ax.set_title("CPU usage for insulin dataset (Pilatus 6M, 100 frames)");

In [ ]:
ax = df.pivot_table(index="nproc", columns="test", values="real_s", aggfunc="mean").plot(marker="o")
ax.set_xlabel("nproc")
ax.set_ylabel("Real time (s)")
ax.set_title("Real time for insulin dataset (Pilatus 6M, 100 frames)");